In [1]:
import math # Matematik kütüphanesi
import torch # Ana Pytorch kütüphanesi
import torch.nn as nn # Sinir ağı katmanları (linear, dropout, vb.)
import torch.nn.functional as F # Aktivasyon fonksiyonları (softmax, relu) için

torch.manual_seed(42) # Tekrarlanabilirlik için rastgelelik tohumu

## Pozisyonel Kodlama (Positional Encoding)

In [2]:
class PositionalEncoding(nn.Module): # Pozisyonel kodlama modülü
  def __init__(self, d_model, max_len=5000): # d_model -> embedding boyutu, max_len: maks. dizi uzunluğu
    super().__init__()
    pe = torch.zeros(max_len, d_model) # (max_len, d_model) boyutunda sıfır matrisi
    position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1) # (max_len, 1) pozisyon indeksleri
    div_term = torch.exp(torch.arange(0, d_model, 2).float()* (-math.log(10000.0)/ d_model)) # Frekans terimleri
    pe[:, 0::2] = torch.sin(position * div_term) # Çift indekslere sinüs
    pe[:, 1::2] = torch.cos(position * div_term) # Tek indekslere kosinüs
    self.register_buffer("pe", pe.unsqueeze(0)) # (1, max_len, d_model) - sabit olarak kaydey

  def forward(self, x): # x: (batch, seq_len, d_model)
    return x + self.pe[:, : x.size(1), :] # Token embedding'ine pozisyon kodlamasını ekle

## Çok Başlı Dikkat Mekanizması (Multi-Head Attention)

In [3]:
class MultiHeadAttention(nn.Module): # Çok başlı dikkat mekanizması
  def __init__(self, d_model, num_heads): # d_model: giriş boyutu, num_heads: baş sayısı
    super().__init__()
    assert d_model % num_heads == 0 # d_model baş sayısına tam bölünmeli
    self.d_model = d_model
    self.num_heads = num_heads
    self.d_k = d_model // num_heads # Her başın alt uzay boyutu

    self.W_q = nn.Linear(d_model, d_model) # Query dönüşüm matrisi
    self.W_k = nn.Linear(d_model, d_model) # Key dönüşüm matrisi
    self.W_v = nn.Linear(d_model, d_model) # Value dönüşüm matrisi
    self.W_o = nn.Linear(d_model, d_model) # Çıkış dönüşüm matrisi

  def split_heads(self,x): # Başlara ayır: (batch, seq, d_model) -> (batch, n_heads, seq, d_k)
    batch_size, seq_len, _ = x.size()
    return x.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)

  def combine_heads(self, x): # Başları birleştir: (batch, n_heads, seq, d_k) -> (batch, seq, d_model)
    batch_size, _, seq_len, _ = x.size()
    return x.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)

  def forward(self, query, key, value, mask=None): # Q, K, V ve isteğe bağlı maske
    Q = self.split_heads(self.W_q(query)) # Query'i başlara ayır
    K = self.split_heads(self.W_k(key)) # Key'i başlara ayır
    V = self.split_heads(self.W_v(value)) # Value'yu başlara ayır

    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k) # Ölçeklendirilmiş dikkat skorları
    if mask is not None:
      scores = scores.masked_fill(mask == 0, float("-inf")) # Maskelenmiş yerlere -inf

    attn = F.softmax(scores, dim=-1) # Softmax ile ağırlıklar
    context = torch.matmul(attn, V) # Ağırlıklı toplam (bağlam vektörü)
    output = self.W_o(self.combine_heads(context)) # Başları birleştir ve çıkışa yansıt
    return output, attn # Çıktı ve dikkat ağırlıkları

## İleri Beslemeli Ağ (Position-wise Feed Forward)

In [4]:
class PositionwiseFeedForward(nn.Module): # Pozisyon bazlı ileri beslemeli ağ
  def __init__(self, d_model, d_ff, dropout=0.1): #d_model: giriş/çıkış, d_ff: gizli katman boyutu
    super().__init__()
    self.linear1 = nn.Linear(d_model, d_ff) # Genişletme katmanı (d_model -> d_ff)
    self.linear2 = nn.Linear(d_ff, d_model) # Daraltma katmanı (d_ff -> d_model)
    self.dropout = nn.Dropout(dropout) # Düzenlileştirme (overfitting önleme)

  def forward(self, x):
    return self.linear2(self.dropout(F.relu(self.linear1(x)))) # Linear -> ReLu -> Dropout -> Linear

## Encoder Katmanı

In [5]:
class EncoderLayer(nn.Module): # Bir encoder katmanı
  def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
    super().__init__()
    self.self_attn = MultiHeadAttention(d_model, num_heads) # Self attention alt katmanı
    self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout) # FFN alt katmanı
    self.norm1 = nn.LayerNorm(d_model) # Layer normalization 1
    self.norm2 = nn.LayerNorm(d_model) # Layer normalization 2
    self.dropout = nn.Dropout(dropout) # Dropout

  def forward(self, x, mask=None): # x: (batch, seq_len, d_model)
    attn_out, _ = self.self_attn(x, x, x, mask) # Self attebtion (Q=K=V=x)
    x = self.norm1(x + self.dropout(attn_out)) # Residual bağlantı + Dropout + LayerNorm
    ff_out = self.feed_forward(x) # FFN'den geçir
    x = self.norm2(x + self.dropout(ff_out)) # Residual bağlantı + Dropout + LayerNorm
    return x

## Decoder Katmanı

In [6]:
class DecoderLayer(nn.Module): # Bir decoder katmanı
  def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
    super().__init__()
    self.self_attn = MultiHeadAttention(d_model, num_heads) # Maskeli self attention
    self.cross_attn = MultiHeadAttention(d_model, num_heads) # Cross attention (encoder'a bakar)
    self.feed_forward = PositionwiseFeedForward(d_model, d_ff, dropout) # FFN
    self.norm1 = nn.LayerNorm(d_model) # Layernorm1
    self.norm2 = nn.LayerNorm(d_model) # Layernorm2
    self.norm3 = nn.LayerNorm(d_model) # Layernorm3
    self.dropout = nn.Dropout(dropout) # Dropout

  def forward(self, x, enc_output, src_mask=None, tgt_mask= None):
    attn_out, _ = self.self_attn(x, x, x, tgt_mask) # Maskeli self attention
    x = self.norm1(x + self.dropout(attn_out)) # Residual + dropout + layernorm

    cross_out, _ = self.cross_attn(x, enc_output, enc_output, src_mask) # Cross attention (q:decoder, K=V:encoder)
    x = self.norm2(x + self.dropout(cross_out)) # Residual + dropout + layernorm

    ff_out = self.feed_forward(x) # FFN'den geçir
    x = self.norm3(x + self.dropout(ff_out)) # Residual + dropout + layernorm
    return x

## Tam Transformer Modeli

In [7]:
class Transformer(nn.Module):
  def __init__(self, src_vocab_size, tgt_vocab_size, d_model=512, num_heads=8,
               num_layers=6, d_ff=2048, max_len=100, dropout=0.1):
    super().__init__()
    self.encoder_embedding = nn.Embedding(src_vocab_size, d_model) # Kaynak token embedding
    self.decoder_embedding = nn.Embedding(tgt_vocab_size, d_model) # Hedef token embedding
    self.positional_encoding = PositionalEncoding(d_model, max_len) # Pozisyonel Kodlama

    self.encoder_layers = nn.ModuleList( # N adet encoder katmanı
        [EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
    )
    self.decoder_layers = nn.ModuleList( # N adet decoder katmanı
        [DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
    )

    self.fc_out = nn.Linear(d_model, tgt_vocab_size) # Çıkış projeksiyonu (vocab üzerinde logit)
    self.dropout = nn.Dropout(dropout) # Dropout

  def generate_mask(self, src, tgt): # Kaynak ve hedef maskelerini oluştur
    src_mask = (src != 0).unsqueeze(1).unsqueeze(2) # (batch, 1, 1, src_len) padding makinesi
    tgt_mask = (tgt != 0).unsqueeze(1).unsqueeze(3) # (batch, 1, tgt_len, 1) padding makinesi

    seq_len = tgt.size(1)
    nopeak_mask = (1 - torch.triu(torch.ones(1, seq_len, seq_len), diagonal=1)).bool() # Üçgen maske
    tgt_mask = tgt_mask & nopeak_mask
    return src_mask, tgt_mask

  def encode(self, src, src_mask):  # Encoder ileri beslemesi
    x = self.dropout(self.positional_encoding(self.encoder_embedding(src))) # Embed + PosEncoding + Dropout
    for layer in self.encoder_layers:
      x = layer(x, src_mask)
    return x

  def decode(self, tgt, enc_output, src_mask, tgt_mask): # Decoder ileri belsmelesi
    x = self.dropout(self.positional_encoding(self.decoder_embedding(tgt))) # Embed + PosEncoding + Dropout
    for layer in self.decoder_layers: # Her decoder katmanından geçir
      x = layer(x, enc_output, src_mask, tgt_mask)
    return x # batch, tgt_len, d_model)

  def forward(self,src, tgt): # Eğitim için ileri besleme
    src_mask, tgt_mask = self.generate_mask(src, tgt) # Maskeleri oluştur
    enc_output = self.encode(src, src_mask) # Encoder çıktısı
    dec_output = self.decode(tgt, enc_output, src_mask, tgt_mask) # Decoder çıktısı
    return self.fc_out(dec_output) # Kelime dağarcığı üzerinde logit'ler



## Modeli Test Etme

In [8]:
src_vocab_size = 1000   # Kaynak kelime dağarcığı boyutu
tgt_vocab_size = 1000   # Hedef kelime dağarcığı boyutu

model = Transformer(
    src_vocab_size, tgt_vocab_size,
    d_model=128, num_heads=4, num_layers=2, d_ff=256, max_len=50
)

src = torch.randint(1, src_vocab_size, (2, 10))
tgt = torch.randint(1, tgt_vocab_size, (2, 12))

out = model(src, tgt)
print("Cikis sekli:", out.shape)

total_params = sum(p.numel() for p in model.parameters())
print(f"Toplam parametre sayisi: {total_params:,}")

Cikis sekli: torch.Size([2, 12, 1000])
Toplam parametre sayisi: 1,047,528
